In [1]:
!pip install xgboost pmdarima prophet --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 12.4 MB/s eta 0:00:00


In [18]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
from prophet import Prophet
import warnings
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

m = pd.read_csv("/content/sample_data/dim_date.csv", parse_dates=['Date'])

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===========================
# Load Dimension Tables
# ===========================

dim_date = pd.read_csv("/content/sample_data/dim_date.csv")
dim_weather = pd.read_csv("/content/sample_data/dim_weather.csv")
dim_lighting = pd.read_csv("/content/sample_data/dim_lighting.csv")
dim_crash_type = pd.read_csv("/content/sample_data/dim_crash_type.csv")
dim_road = pd.read_csv("/content/sample_data/dim_road.csv")
dim_time = pd.read_csv("/content/sample_data/dim_time.csv")
dim_damage = pd.read_csv("/content/sample_data/dim_damage.csv")
dim_severity = pd.read_csv("/content/sample_data/dim_severity.csv")

# ===========================
# Load Fact Table
# ===========================

fact_accidents = pd.read_csv("/content/sample_data/fact_accidents.csv")

# ===========================
# Create Star Schema
# ===========================

df = (fact_accidents
    .merge(dim_date, on='Date_Key', how='left')
    .merge(dim_weather, on='Weather_Key', how='left')
    .merge(dim_lighting, on='Lighting_Key', how='left')
    .merge(dim_crash_type, on='Crash_Type_Key', how='left')
    .merge(dim_road, on='Road_Key', how='left')
    .merge(dim_time, on='Time_Key', how='left')
    .merge(dim_damage, on='Damage_Key', how='left')
    .merge(dim_severity, on='Severity_Key', how='left')
)

df["Date"] = pd.to_datetime(df["Date"])

df.head()

,Accident_Key,Date_Key,Weather_Key,Lighting_Key,Crash_Type_Key,Road_Key,Time_Key,Damage_Key,Severity_Key,Number_of_vehicles_involved,...,roadway_surface_cond,Road_alignment,trafficway_type,road_defect,crash_hour,AM/PM,Time Buckets,Cost_of_damage,Damage_level,Crash_severity
0,1,2918,1,1,1,1,1,1,1,2,...,UNKNOWN,STRAIGHT AND LEVEL,NOT DIVIDED,UNKNOWN,13,PM,Afternoon,"$501 - $1,500",2,No Injury
1,2,2933,1,2,1,2,2,2,1,2,...,DRY,STRAIGHT AND LEVEL,FOUR WAY,NO DEFECTS,0,AM,Late Night,"OVER $1,500",3,No Injury
2,3,2321,1,1,1,3,3,1,1,3,...,DRY,STRAIGHT AND LEVEL,T-INTERSECTION,NO DEFECTS,10,AM,Morning,"$501 - $1,500",2,No Injury
3,4,2929,1,1,2,2,4,2,2,2,...,DRY,STRAIGHT AND LEVEL,FOUR WAY,NO DEFECTS,19,PM,Evening,"OVER $1,500",3,Moderate
4,5,2939,1,1,1,4,5,1,1,2,...,UNKNOWN,STRAIGHT AND LEVEL,T-INTERSECTION,UNKNOWN,14,PM,Afternoon,"$501 - $1,500",2,No Injury


# ================= FEATURE ENGINEERING (for XGBoost) =================

In [4]:
monthly = (
    df.groupby(pd.Grouper(key="Date", freq="MS"))
      .agg(
          Accident_Count=("Accident_Key", "count"),
          Total_Injuries=("Total_injuries_num", "sum"),
          Fatalities=("Fatal_injuries_num", "sum")
      )
      .reset_index()
)

# ================= TIME-BASED SPLIT =================
# Train: 2018-01 to 2023-12 | Test: 2024 (12 months) -- no random shuffling

In [5]:
feat = monthly.copy()

feat["year"] = feat["Date"].dt.year
feat["month"] = feat["Date"].dt.month
feat["quarter"] = feat["Date"].dt.quarter

feat["month_sin"] = np.sin(2 * np.pi * feat["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * feat["month"] / 12)

feat["t"] = np.arange(len(feat))

# Lag features
for lag in [1, 2, 3, 12]:
    feat[f"lag_{lag}"] = feat["Accident_Count"].shift(lag)

# Rolling averages
feat["roll3"] = feat["Accident_Count"].shift(1).rolling(3).mean()
feat["roll12"] = feat["Accident_Count"].shift(1).rolling(12).mean()

feat = feat.dropna().reset_index(drop=True)

In [6]:
train = feat[feat['Date'] < '2024-01-01']
test = feat[feat['Date'] >= '2024-01-01']

feature_cols = ['year','month','quarter','month_sin','month_cos','t',
                 'lag_1','lag_2','lag_3','lag_12','roll3','roll12']

X_train = train[feature_cols]
y_train = train["Accident_Count"]

X_test = test[feature_cols]
y_test = test["Accident_Count"]


# ================= MODEL 1: XGBOOST =================

In [7]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_r2 = r2_score(y_test, xgb_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = mean_squared_error(y_test, xgb_pred) ** 0.5
xgb_mape = np.mean(np.abs((y_test.values - xgb_pred) / y_test.values)) * 100

print("=== XGBoost (test = 2024) ===")
print(f"R2: {xgb_r2:.3f}  MAE: {xgb_mae:.1f}  RMSE: {xgb_rmse:.1f}  MAPE: {xgb_mape:.2f}%")


=== XGBoost (test = 2024) ===
R2: 0.261  MAE: 160.3  RMSE: 299.3  MAPE: 11.27%


# ================= MODEL 2: PROPHET =================

In [8]:
from prophet import Prophet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

# Prepare Prophet data
pdf = monthly[['Date', 'Accident_Count']].copy()
pdf.columns = ['ds', 'y']

# Train/Test Split
pdf_train = pdf[pdf['ds'] < '2024-01-01']
pdf_test = pdf[pdf['ds'] >= '2024-01-01']

# Build Prophet model
prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

prophet_model.fit(pdf_train)

# Forecast only the test period
future = prophet_model.make_future_dataframe(
    periods=len(pdf_test),
    freq='MS'
)

forecast = prophet_model.predict(future)

# Extract predictions for test dates
prophet_test_pred = (
    forecast.set_index("ds")
            .loc[pdf_test["ds"], "yhat"]
            .values
)

# Evaluation
prophet_r2 = r2_score(pdf_test["y"], prophet_test_pred)
prophet_mae = mean_absolute_error(pdf_test["y"], prophet_test_pred)
prophet_rmse = np.sqrt(mean_squared_error(pdf_test["y"], prophet_test_pred))
prophet_mape = np.mean(
    np.abs((pdf_test["y"] - prophet_test_pred) / pdf_test["y"])
) * 100

print("="*40)
print("Prophet Results")
print("="*40)
print(f"R²   : {prophet_r2:.4f}")
print(f"MAE  : {prophet_mae:.2f}")
print(f"RMSE : {prophet_rmse:.2f}")
print(f"MAPE : {prophet_mape:.2f}%")

Prophet Results
R²   : -0.0196
MAE  : 185.76
RMSE : 351.44
MAPE : 13.36%


# ================= PICK BEST MODEL =================

In [9]:
results = pd.DataFrame({
    'Model': ['XGBoost', 'Prophet'],
    'R2': [xgb_r2, prophet_r2],
    'MAE': [xgb_mae, prophet_mae],
    'RMSE': [xgb_rmse, prophet_rmse],
    'MAPE_%': [xgb_mape, prophet_mape]
})
print("\n=== Model comparison ===")
print(results.to_string(index=False))



=== Model comparison ===
  Model        R2        MAE       RMSE    MAPE_%
XGBoost  0.260577 160.277649 299.278377 11.270835
Prophet -0.019613 185.758418 351.436247 13.359794


================= FINAL MODEL: REFIT ON ALL STABLE DATA & FORECAST 24 MONTHS =================

In [10]:
final_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Train on all available data
final_model.fit(feat[feature_cols], feat['Accident_Count'])

# Historical values
history = list(feat['Accident_Count'].values)

last_date = feat['Date'].max()

future_dates = pd.date_range(
    last_date + pd.offsets.MonthBegin(1),
    periods=24,
    freq='MS'
)

preds = []

for i, d in enumerate(future_dates):

    t_val = len(feat) + i

    row = {
        'year': d.year,
        'month': d.month,
        'quarter': (d.month-1)//3 + 1,
        'month_sin': np.sin(2*np.pi*d.month/12),
        'month_cos': np.cos(2*np.pi*d.month/12),
        't': t_val,

        'lag_1': history[-1],
        'lag_2': history[-2],
        'lag_3': history[-3],
        'lag_12': history[-12],

        'roll3': np.mean(history[-3:]),
        'roll12': np.mean(history[-12:])
    }

    X_row = pd.DataFrame([row])[feature_cols]

    pred = final_model.predict(X_row)[0]

    preds.append(pred)

    history.append(pred)

forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Forecast_Accidents': np.round(preds).astype(int)
})

forecast_df.to_csv("forecast_2026_2027.csv", index=False)

print(forecast_df.to_string(index=False))

print("\nTotal forecast year 1:",
      forecast_df.iloc[:12]['Forecast_Accidents'].sum())

print("Total forecast year 2:",
      forecast_df.iloc[12:]['Forecast_Accidents'].sum())

      Date  Forecast_Accidents
2025-02-01                1176
2025-03-01                1330
2025-04-01                1413
2025-05-01                1577
2025-06-01                1624
2025-07-01                1523
2025-08-01                1589
2025-09-01                1557
2025-10-01                1541
2025-11-01                1457
2025-12-01                1493
2026-01-01                1319
2026-02-01                1311
2026-03-01                1349
2026-04-01                1388
2026-05-01                1517
2026-06-01                1577
2026-07-01                1528
2026-08-01                1545
2026-09-01                1547
2026-10-01                1545
2026-11-01                1459
2026-12-01                1495
2027-01-01                1319

Total forecast year 1: 17599
Total forecast year 2: 17580


================= FINAL MODEL: Picture and download csv file ALL STABLE DATA & FORECAST 24 MONTHS =================

In [11]:

plt.figure(figsize=(13,5))

plt.plot(
    feat['Date'],
    feat['Accident_Count'],
    label='Historical',
    color='#2563eb'
)

plt.plot(
    forecast_df['Date'],
    forecast_df['Forecast_Accidents'],
    label='Forecast (Next 24 Months)',
    color='#dc2626',
    linestyle='--'
)

plt.axvline(last_date, color='gray', linestyle=':')

plt.title('Monthly Traffic Accidents: Historical + 24-Month Forecast')
plt.xlabel('Date')
plt.ylabel('Accident Count')

plt.legend()
plt.tight_layout()

plt.savefig('forecast_plot.png', dpi=130)

plt.show()

print("\nForecast saved to forecast_2026_2027.csv")
print("Plot saved to forecast_plot.png")


Forecast saved to forecast_2026_2027.csv
Plot saved to forecast_plot.png


# PREDICTED INJURY RATE FOR NEXT TWO YEARS  (XGBoost time-series regressor)

In [12]:
def build_features(df, target_col):
    data = df.copy()

    data['year'] = data['Date'].dt.year
    data['month'] = data['Date'].dt.month
    data['quarter'] = data['Date'].dt.quarter

    # Lag features
    data['lag1'] = data[target_col].shift(1)
    data['lag2'] = data[target_col].shift(2)
    data['lag3'] = data[target_col].shift(3)

    # Rolling averages
    data['roll3'] = data[target_col].shift(1).rolling(3).mean()
    data['roll6'] = data[target_col].shift(1).rolling(6).mean()

    data = data.dropna().reset_index(drop=True)

    return data

In [13]:
feat_cols = [
    'year',
    'month',
    'quarter',
    'lag1',
    'lag2',
    'lag3',
    'roll3',
    'roll6'
]

In [14]:
def recursive_forecast(model, history, target_col, periods):

    history = history.copy()

    forecasts = []

    for i in range(periods):

        next_date = history['Date'].max() + pd.DateOffset(months=1)

        lag1 = history[target_col].iloc[-1]
        lag2 = history[target_col].iloc[-2]
        lag3 = history[target_col].iloc[-3]

        roll3 = history[target_col].iloc[-3:].mean()
        roll6 = history[target_col].iloc[-6:].mean()

        X = pd.DataFrame({
            'year':[next_date.year],
            'month':[next_date.month],
            'quarter':[next_date.quarter],
            'lag1':[lag1],
            'lag2':[lag2],
            'lag3':[lag3],
            'roll3':[roll3],
            'roll6':[roll6]
        })

        pred = model.predict(X)[0]

        forecasts.append({
            'Date': next_date,
            target_col: pred
        })

        history = pd.concat([
            history,
            pd.DataFrame({
                'Date':[next_date],
                target_col:[pred]
            })
        ], ignore_index=True)

    return pd.DataFrame(forecasts)

In [15]:
print("\n" + "="*70)
print("INJURY RATE FORECAST - NEXT 24 MONTHS")
print("="*70)

monthly_inj = df.groupby(pd.Grouper(key='Date', freq='MS')).agg(
    accident_count=('Accident_Key', 'count'),
    total_injuries=('Total_injuries_num', 'sum')
).reset_index()
monthly_inj['injury_rate'] = monthly_inj['total_injuries'] / monthly_inj['accident_count']

stable_inj = monthly_inj[(monthly_inj['Date'] >= '2018-01-01') &
                          (monthly_inj['Date'] <= '2024-12-01')].reset_index(drop=True)

inj_feat = build_features(stable_inj[['Date', 'injury_rate']], 'injury_rate')
inj_model = xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, random_state=42)

# quick time-based validation
train_i = inj_feat[inj_feat['Date'] < '2024-01-01']
test_i = inj_feat[inj_feat['Date'] >= '2024-01-01']
inj_model.fit(train_i[feat_cols], train_i['injury_rate'])
inj_pred_test = inj_model.predict(test_i[feat_cols])

# refit on full stable history, forecast forward
inj_model.fit(inj_feat[feat_cols], inj_feat['injury_rate'])
inj_forecast = recursive_forecast(inj_model, stable_inj[['Date', 'injury_rate']], 'injury_rate', 24)


inj_forecast['injury_rate'] = inj_forecast['injury_rate'].round(4)
print("\nForecasted monthly injury rate (injuries per accident):")
print(inj_forecast.to_string(index=False))
inj_forecast['injury_rate'] = inj_forecast['injury_rate'].round(4)

inj_forecast.to_csv(
    "injury_rate_forecast.csv",
    index=False
)

print("CSV file saved successfully.")

# Download the file (Google Colab)
from google.colab import files
files.download("injury_rate_forecast.csv")


INJURY RATE FORECAST - NEXT 24 MONTHS

Forecasted monthly injury rate (injuries per accident):
      Date  injury_rate
2025-01-01       0.3645
2025-02-01       0.3653
2025-03-01       0.4171
2025-04-01       0.4358
2025-05-01       0.4539
2025-06-01       0.4847
2025-07-01       0.4854
2025-08-01       0.4587
2025-09-01       0.4640
2025-10-01       0.4648
2025-11-01       0.4258
2025-12-01       0.4043
2026-01-01       0.3631
2026-02-01       0.3521
2026-03-01       0.4157
2026-04-01       0.4359
2026-05-01       0.4470
2026-06-01       0.4779
2026-07-01       0.4858
2026-08-01       0.4726
2026-09-01       0.4606
2026-10-01       0.4646
2026-11-01       0.4402
2026-12-01       0.4090
CSV file saved successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Predicted Crash Severity FOR NEXT TWO YEARS (XGBoost time-series regressor)

In [19]:
sev_df = df[['Weather', 'Crash_severity']].dropna().copy()

weather_encoder = LabelEncoder()
severity_encoder = LabelEncoder()

sev_df["Weather_Encoded"] = weather_encoder.fit_transform(sev_df["Weather"])
sev_df["Severity_Encoded"] = severity_encoder.fit_transform(sev_df["Crash_severity"])

X = sev_df[["Weather_Encoded"]]
y = sev_df["Severity_Encoded"]

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(severity_encoder.classes_),
    random_state=42,
    eval_metric="mlogloss"
)

model.fit(X, y)

sev_df["Predicted_Crash_Severity"] = severity_encoder.inverse_transform(
    model.predict(X)
)

severity_weather_table = (
    sev_df.groupby(["Weather", "Predicted_Crash_Severity"])
          .size()
          .reset_index(name="Predicted_Crash_Count")
)

print(severity_weather_table)

severity_weather_table.to_csv(
    "Predicted_Crash_Severity_by_Weather.csv",
    index=False
)

print("Saved: Predicted_Crash_Severity_by_Weather.csv")

                     Weather Predicted_Crash_Severity  Predicted_Crash_Count
0   BLOWING SAND, SOIL, DIRT                No Injury                      1
1               BLOWING SNOW                No Injury                    127
2                      CLEAR                No Injury                 164700
3            CLOUDY/OVERCAST                No Injury                   7533
4             FOG/SMOKE/HAZE                No Injury                    360
5      FREEZING RAIN/DRIZZLE                No Injury                    510
6                      OTHER                No Injury                    627
7                       RAIN                No Injury                  21703
8     SEVERE CROSS WIND GATE                No Injury                     32
9                 SLEET/HAIL                No Injury                    308
10                      SNOW                No Injury                   6871
11                   UNKNOWN                No Injury                   6534

# Predicted Damage Level by Road Surface FOR NEXT TWO YEARS (XGBoost time-series regressor)

In [20]:
damage_df = df[['roadway_surface_cond', 'Damage_level']].dropna().copy()

road_encoder = LabelEncoder()
damage_encoder = LabelEncoder()

damage_df["Road_Encoded"] = road_encoder.fit_transform(
    damage_df["roadway_surface_cond"]
)

damage_df["Damage_Encoded"] = damage_encoder.fit_transform(
    damage_df["Damage_level"]
)

X = damage_df[["Road_Encoded"]]
y = damage_df["Damage_Encoded"]

damage_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    objective="multi:softmax",
    num_class=len(damage_encoder.classes_),
    random_state=42,
    eval_metric="mlogloss"
)

damage_model.fit(X, y)

damage_df["Predicted_Damage_Level"] = damage_encoder.inverse_transform(
    damage_model.predict(X)
)

damage_surface_table = (
    damage_df.groupby(
        ["roadway_surface_cond", "Predicted_Damage_Level"]
    )
    .size()
    .reset_index(name="Predicted_Crash_Count")
)

print(damage_surface_table)

damage_surface_table.to_csv(
    "Predicted_Damage_Level_by_Road_Surface.csv",
    index=False
)

print("Saved: Predicted_Damage_Level_by_Road_Surface.csv")

  roadway_surface_cond  Predicted_Damage_Level  Predicted_Crash_Count
0                  DRY                       3                 155905
1                  ICE                       3                   1303
2                OTHER                       3                    438
3      SAND, MUD, DIRT                       3                     40
4        SNOW OR SLUSH                       3                   6203
5              UNKNOWN                       3                  12509
6                  WET                       3                  32908
Saved: Predicted_Damage_Level_by_Road_Surface.csv


In [21]:
from google.colab import files

files.download("Predicted_Crash_Severity_by_Weather.csv")
files.download("Predicted_Damage_Level_by_Road_Surface.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Marge in one file

In [22]:
# Define the file paths and the names you want for each sheet
files_to_combine = {
    "Injury Rate Forecast": "injury_rate_forecast.csv",
    "Forecast 2026-2027": "forecast_2026_2027.csv",
    "Severity by Weather": "Predicted_Crash_Severity_by_Weather.csv",
    "Damage by Road Surface": "Predicted_Damage_Level_by_Road_Surface.csv",
}

# Create an Excel writer object
output_file = "Accident_Analysis_Consolidated_Data.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, csv_path in files_to_combine.items():
        try:
            # Read the CSV file
            df = pd.read_csv(csv_path)
            # Write to a specific sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Successfully added '{sheet_name}' to the workbook.")
        except FileNotFoundError:
            print(f"Warning: File {csv_path} not found. Skipping...")

print(f"\nAll done! Your consolidated file is saved as: {output_file}")

Successfully added 'Injury Rate Forecast' to the workbook.
Successfully added 'Forecast 2026-2027' to the workbook.
Successfully added 'Severity by Weather' to the workbook.
Successfully added 'Damage by Road Surface' to the workbook.

All done! Your consolidated file is saved as: Accident_Analysis_Consolidated_Data.xlsx
